## Setup
Run notebooks 01–07 first. Install dependencies once from the project root with `python -m pip install -r requirements.txt`.

In [1]:
from pathlib import Path
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate/"data").is_dir() and (candidate/"notebooks").is_dir(): PROJECT_ROOT=candidate; break
else: raise RuntimeError("Start Jupyter inside the project so its root can be found.")
DATA_DIR=PROJECT_ROOT/"data"; RAW_DATA_DIR=DATA_DIR/"01_raw"; INTERMEDIATE_DIR=DATA_DIR/"02_intermediate"
MODEL_DIR=PROJECT_ROOT/"models"; REPORT_DIR=PROJECT_ROOT/"reports"
INTERMEDIATE_DIR.mkdir(parents=True,exist_ok=True); MODEL_DIR.mkdir(parents=True,exist_ok=True); REPORT_DIR.mkdir(parents=True,exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")

Project root: c:\Users\acer\Desktop\Programs\Projects_for _data_science\Fraud_Detection_Model


# 08 · Risk scoring

A model probability is a ranking signal. A useful risk score also needs a clear scale, calibration checks, and documented limits. This notebook loads the selected artifact, reproduces the chronological validation split, scores it, and creates illustrative risk bands. The bands are examples for analysis, not universal operating thresholds.

## 1. Load engineered data and the selected model

### Why load the artifact and its feature list

The model artifact is the source of truth for the estimator, feature order, and any fitted preprocessing. Reusing that list prevents a subtle but serious error: a model receiving the right values in the wrong columns. The engineered training data is needed here because it contains the labels used to inspect validation behavior.

In [ ]:
feature_path = INTERMEDIATE_DIR / "train_features.pkl"
artifact_path = MODEL_DIR / "best_experiment_model.joblib"

if not feature_path.exists() or not artifact_path.exists():
    raise FileNotFoundError("Run notebooks 04 and 06 first.")

train_fe = pd.read_pickle(feature_path)
artifact = joblib.load(artifact_path)
feature_cols = list(artifact["feature_cols"])
missing = sorted(set(feature_cols) - set(train_fe.columns))
if missing:
    raise ValueError(f"Feature data lacks model inputs: {missing[:10]}")

print(f"Model: {artifact.get('experiment_name', type(artifact['model']).__name__)}")
print(f"Feature count: {len(feature_cols)} | Rows: {len(train_fe):,}")

## 2. Rebuild the chronological validation window

The latest 20% of training events form validation, matching notebooks 05–07. The same window influenced model selection and threshold choices, so these numbers are exploratory rather than an unbiased final estimate.

### Why this validation slice and these metrics

The latest 20% simulates scoring a later period, which is closer to deployment than a random split. ROC-AUC measures ranking across thresholds; average precision summarizes positive-class precision/recall and is more informative when fraud is rare. This is the same window used to choose the model, so the numbers are exploratory and optimistic after selection.

In [ ]:
ordered = train_fe.sort_values("TransactionDT", kind="stable").reset_index(drop=True)
split_index = int(len(ordered) * 0.8)
validation = ordered.iloc[split_index:].copy()

X_validation = validation[feature_cols].replace([np.inf, -np.inf], np.nan)
y_validation = validation["isFraud"].astype(int)
model = artifact["model"]

# Reuse exactly the preprocessing saved with this model.
imputer = artifact.get("imputer")
scaler = artifact.get("scaler")
if imputer is not None:
    X_for_model = imputer.transform(X_validation)
else:
    X_for_model = X_validation
if scaler is not None:
    X_for_model = scaler.transform(X_for_model)

fraud_probability = model.predict_proba(X_for_model)[:, 1]
print(f"Validation rows: {len(y_validation):,}")
print(f"Observed fraud prevalence: {y_validation.mean():.3%}")
print(f"ROC-AUC (ranking): {roc_auc_score(y_validation, fraud_probability):.4f}")
print(f"Average precision (rare-class ranking): {average_precision_score(y_validation, fraud_probability):.4f}")

## 3. Convert probability to a readable score

The 0–100 score is the model output multiplied by 100. It does not make an uncalibrated model calibrated. Check notebook 07’s reliability curve before describing it as a literal probability.

### Why create a score from probability

The 0–100 scale is a display transformation only: `risk_score = 100 × model_probability`. It preserves ordering and adds no predictive information. It is not a calibrated percentage unless the model’s probabilities match observed rates on a representative, mature-label sample.

In [ ]:
display_columns = [
    column for column in ["TransactionID", "TransactionDT", "TransactionAmt", "isFraud"]
    if column in validation.columns
]
scored = validation[display_columns].copy()
scored["fraud_probability"] = fraud_probability
scored["risk_score"] = (100 * fraud_probability).round(2)
scored.head()

## 4. Use illustrative bands and inspect observed outcomes

### Why make bands, and why these cutoffs are not recommendations

Bands simplify routing and discussion, but every boundary creates a policy trade-off: a lower boundary moves more transactions into higher-risk handling; a higher boundary reduces workload while leaving more fraud in lower bands. The 5% and 20% values are examples only. Choose production bands using validated costs, review capacity, and customer impact.

In [ ]:
# Illustrative boundaries only; do not treat these as approved policy.
LOWER_BAND_MAX = 0.05
ELEVATED_BAND_MAX = 0.20

scored["risk_band"] = pd.cut(
    scored["fraud_probability"],
    bins=[-np.inf, LOWER_BAND_MAX, ELEVATED_BAND_MAX, np.inf],
    labels=["lower", "elevated", "high"],
    right=False,
)

band_summary = scored.groupby("risk_band", observed=False).agg(
    transactions=("risk_score", "size"),
    mean_score=("risk_score", "mean"),
    observed_fraud_rate=("isFraud", "mean"),
    fraud_cases=("isFraud", "sum"),
)
band_summary

### What the plots add

The histogram shows how much traffic falls at each score; the bar plot checks whether observed fraud rates rise across bands. A band with few cases can have a noisy fraud rate, and a monotone pattern on this one validation sample does not guarantee future calibration.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].hist(scored["risk_score"], bins=30, color="slateblue")
axes[0].set(
    title="Validation score distribution",
    xlabel="Score (0–100)",
    ylabel="Transactions",
)
band_summary["observed_fraud_rate"].plot.bar(ax=axes[1], color="firebrick")
axes[1].set(
    title="Observed fraud rate by illustrative band",
    xlabel="Risk band",
    ylabel="Fraud rate",
)
plt.tight_layout()
plt.show()

## 5. Interpretation limits

- Historical benchmark output is not a guarantee about a future transaction.
- Labels may arrive late or be incomplete, changing measured fraud rates.
- Cost, review capacity, customer impact, and calibration matter when choosing actions.
- These engineered velocity features are offline. Live inference needs stateful features using only past events.